# 🚀 Imperative Migration Example: Past the Clean Baseline

This notebook demonstrates how to perform a **Manual Migration** using the high-integrity output of the `clean-dataset` CLI. 

In this approach, we ignore the `config.yaml` pipeline and use standard vectorized Pandas operations to perform domain-specific cleaning. This is the recommended path for users who want total control over their transformation sequence.

In [1]:
import pandas as pd
import numpy as np
from dd_cleaner.notebook_utils import init_notebook_session, get_cleaned_data

# 1. Initialize session and load the 'Clean Baseline'
# Point to the directory containing the 'data/' folder
coord, _ = init_notebook_session("..")
df = get_cleaned_data(coord)

print(f"Loaded Baseline: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

✅ Notebook session initialized for workspace: /home/rajiv/programming/dd_parser_cleaner/tests

Available Artifacts:

Artifact Name                           File Name  \
0                         Raw Data                   sba_loans_raw.csv   
1                     Cleaned Data             sba_loans_raw_clean.csv   
2             Tagged Entities (DD)  sba_loans_raw_analysis_results.csv   
3  Cleaning Recommendations Report         cleaning_recommendations.md   
4                 Profiling Report   sba_loans_raw_profiling_report.md   
5                   Handshake File         parser_cleaner_handshake.md   
6                  Quarantine File        sba_loans_raw_quarantine.csv   

                                            Location  Exists  
0                             data/sba_loans_raw.csv    True  
1            data/dd_cleaner/sba_loans_raw_clean.csv    True  
2  documents/dd_analysis_results/sba_loans_raw_an...    True  
3   documents/dd_cleaner/cleaning_recommendations.md    True  
4  documents/dd_cleaner/sba_loans_raw_profiling_r...    True  
5   documents/dd_cleaner/parser_cleaner_handshake.md    True  
6       data/quarantine/sba_loans_raw_quarantine.csv   False

Loaded Baseline: 116345 rows, 31 columns


,asofdate,program,locationid,borrname,borrstreet,borrcity,borrstate,borrzip,grossapproval,approvaldate,...,sbadistrictoffice,congressionaldistrict,businesstype,businessage,loanstatus,paidinfulldate,chargeoffdate,grosschargeoffamount,jobssupported,collateralind
0,3/31/2026,504,188194,"Sai Ram Hospitality, Inc.",1010 E. Columbian Blvd..,Litchfield,IL,62056,810000,10/2/2009,...,ILLINOIS DISTRICT OFFICE,13.0,CORPORATION,Less than 4 years old but at least 3,PREPAID IN FULL,9/30/2016,NaN,0.0,0,True
1,3/31/2026,504,188242,Martin Harper P.C.,3 E. Ramona Avenue.,Colorado springs,CO,80905,127000,10/2/2009,...,COLORADO DISTRICT OFFICE,5.0,CORPORATION,Less than 4 years old but at least 3,PREPAID IN FULL,11/30/2020,NaN,0.0,1,True
2,3/31/2026,504,613875,Patchwerk Recording Studios,1094 Hemphill Avenue.,Atlanta,GA,30318,1287000,10/2/2009,...,GEORGIA DISTRICT OFFICE,5.0,CORPORATION,Less than 4 years old but at least 3,PREPAID IN FULL,4/30/2018,NaN,0.0,5,True
3,3/31/2026,504,188333,HAMMES SEED INC,1123 120th Rd..,Seneca,KS,66538,273000,10/2/2009,...,KANSAS CITY DISTRICT OFFICE,2.0,CORPORATION,Less than 4 years old but at least 3,CURRENT,NaN,NaN,0.0,1,True
4,3/31/2026,504,188210,"PriceKubecka, PLLC",465 West President George Bush.,Richardson,TX,75080,1500000,10/2/2009,...,DALLAS / FT WORTH DISTRICT OFFICE,3.0,CORPORATION,Less than 4 years old but at least 3,CANCELED,NaN,NaN,0.0,13,True


## 1. Row-Transformation (Filtering)
Removing records based on domain-specific logic (e.g., date consistency).

In [2]:
# Example: Remove rows where First Disbursement is after Paid In Full
fdd = pd.to_datetime(df['firstdisbursementdate'], errors='coerce')
pifd = pd.to_datetime(df['paidinfulldate'], errors='coerce')

invalid_mask = fdd > pifd
df = df[~invalid_mask].copy()

print(f"Rows after date consistency filter: {len(df)} (Dropped {invalid_mask.sum()})")

Rows after date consistency filter: 116327 (Dropped 18)


## 2. Attribute Management (Drop & Rename)
Removing technical noise and renaming columns for downstream readability.

In [3]:
# Drop unnecessary technical columns
cols_to_drop = ['asofdate', 'locationid']
df.drop(columns=cols_to_drop, errors='ignore', inplace=True)

# Rename attributes for clarity
rename_map = {
    'borrname': 'borrower_name',
    'grossapproval': 'total_loan_amount'
}
df.rename(columns=rename_map, inplace=True)

print(f"Columns remaining: {list(df.columns[:5])}...")

Columns remaining: ['program', 'borrower_name', 'borrstreet', 'borrcity', 'borrstate']...


## 3. Missing-Values (Imputation)
Filling NaNs with logical defaults based on the attribute's nature.

In [4]:
# Impute categorical with a sentinel string
df['borrstate'] = df['borrstate'].fillna('UNKNOWN')

# Impute numeric with a constant (e.g., 0 for charge-off amount if missing)
df['grosschargeoffamount'] = df['grosschargeoffamount'].fillna(0)

print("Imputation complete.")

Imputation complete.


## 4. Attribute Derivation (Feature Engineering)
Creating new analytical features from existing attributes.

In [5]:
# Derive 'loan_performance_ratio'
df['loan_performance_ratio'] = df['grosschargeoffamount'] / df['total_loan_amount']

# Create a boolean flag for 'is_distressed'
df['is_distressed'] = df['loan_performance_ratio'] > 0

df[['total_loan_amount', 'grosschargeoffamount', 'is_distressed']].head()

,total_loan_amount,grosschargeoffamount,is_distressed
0,810000,0.0,False
1,127000,0.0,False
2,1287000,0.0,False
3,273000,0.0,False
4,1500000,0.0,False


## 5. The Unified Migration Chain (Best Practice Sequence)

To prevent clobbering dependencies, we use the following functional sequence. Note that **Renaming and Dropping** happen last to ensure all preceding logic has access to the full raw feature set.

In [6]:
def apply_migration_pipeline(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    A consolidated pipeline that respects data dependencies.
    """
    work_df = raw_df.copy()
    
    # 1. Row Filtering (Integrity First)
    fdd = pd.to_datetime(work_df['firstdisbursementdate'], errors='coerce')
    pifd = pd.to_datetime(work_df['paidinfulldate'], errors='coerce')
    work_df = work_df[~(fdd > pifd)].copy()
    
    # 2. Imputation (Fix holes before calculation)
    work_df['borrstate'] = work_df['borrstate'].fillna('UNKNOWN')
    work_df['grosschargeoffamount'] = work_df['grosschargeoffamount'].fillna(0)
    
    # 3. Derivation (Logic depends on original column names)
    # Note: We use 'grossapproval' here because it hasn't been renamed yet
    work_df['loan_performance_ratio'] = work_df['grosschargeoffamount'] / work_df['grossapproval']
    work_df['is_distressed'] = work_df['loan_performance_ratio'] > 0
    
    # 4. Schema Management (The terminal 'finishing' act)
    work_df.rename(columns={
        'borrname': 'borrower_name',
        'grossapproval': 'total_loan_amount'
    }, inplace=True)
    
    work_df.drop(columns=['asofdate', 'locationid'], errors='ignore', inplace=True)
    
    return work_df

# Execute the chain
df_raw_baseline = get_cleaned_data(coord)
df_final = apply_migration_pipeline(df_raw_baseline)

print(f"Final Dataset Shape: {df_final.shape}")
df_final[['borrower_name', 'total_loan_amount', 'loan_performance_ratio']].head()

Final Dataset Shape: (116327, 31)


,borrower_name,total_loan_amount,loan_performance_ratio
0,"Sai Ram Hospitality, Inc.",810000,0.0
1,Martin Harper P.C.,127000,0.0
2,Patchwerk Recording Studios,1287000,0.0
3,HAMMES SEED INC,273000,0.0
4,"PriceKubecka, PLLC",1500000,0.0
